# Day 07 — OOP Part 2: Inheritance and ABCs
**Vidya Sankalp · Python for AI Engineering**

## Learning goals
- Understand inheritance, method overriding, and `super()`
- Use an abstract base class (ABC) to define a required interface
- Recognize public, protected, and private naming conventions
- Understand `@classmethod`, `@staticmethod`, and composition

We'll use **Employee** and **Customer** examples, then connect the ideas to service classes used in applications.

## 1. Inheritance: reuse shared behavior

Imagine every employee has a name and an employee ID. A full-time employee also has a monthly salary, while a contractor has an hourly rate.

Instead of repeating common employee details, define them once in a parent class.

In [ ]:
class Employee:
    def __init__(self, name, employee_id):
        self.name = name
        self.employee_id = employee_id

    def show_details(self):
        print(f"{self.name} (ID: {self.employee_id})")

class FullTimeEmployee(Employee):
    def __init__(self, name, employee_id, monthly_salary):
        super().__init__(name, employee_id)
        self.monthly_salary = monthly_salary

class Contractor(Employee):
    def __init__(self, name, employee_id, hourly_rate):
        super().__init__(name, employee_id)
        self.hourly_rate = hourly_rate

employee = FullTimeEmployee("Ananya", 101, 75000)
contractor = Contractor("Rahul", 201, 1200)

employee.show_details()
print("Monthly salary:", employee.monthly_salary)
contractor.show_details()
print("Hourly rate:", contractor.hourly_rate)

### The key ideas

- **Parent/base class:** `Employee` contains shared data and behavior.
- **Child/subclass:** `FullTimeEmployee` and `Contractor` inherit from `Employee`.
- **`super().__init__(...)`:** calls the parent's initializer to set up shared attributes.
- **IS-A relationship:** a `FullTimeEmployee` is an `Employee`.

Without `super()`, the child's initializer would need to set up `name` and `employee_id` itself.

## 2. Method overriding

A child class can replace a parent's method with behavior suited to that child. This is called **overriding**.

In [ ]:
class Employee:
    def __init__(self, name):
        self.name = name

    def calculate_pay(self):
        return 0  # Generic placeholder for this simple example

class FullTimeEmployee(Employee):
    def __init__(self, name, monthly_salary):
        super().__init__(name)
        self.monthly_salary = monthly_salary

    def calculate_pay(self):
        return self.monthly_salary

class Contractor(Employee):
    def __init__(self, name, hours_worked, hourly_rate):
        super().__init__(name)
        self.hours_worked = hours_worked
        self.hourly_rate = hourly_rate

    def calculate_pay(self):
        return self.hours_worked * self.hourly_rate

people = [
    FullTimeEmployee("Ananya", 75000),
    Contractor("Rahul", 120, 1200),
]

for person in people:
    print(f"{person.name}: ₹{person.calculate_pay():,}")

Notice that the same call, `person.calculate_pay()`, produces different results depending on the object's class. This is a basic example of **polymorphism**.

## 3. ABC: an abstract base class

Sometimes every child class **must** provide a method. Python's `abc` module lets us define that contract.

- `ABC` marks a class as an abstract base class.
- `@abstractmethod` marks a method that subclasses must implement.
- You cannot create an instance of an abstract class until all its abstract methods are implemented.

This is useful when multiple classes must follow the same interface.

In [ ]:
from abc import ABC, abstractmethod

class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def calculate_pay(self):
        """Each employee type must define its own pay calculation."""
        pass

class FullTimeEmployee(Employee):
    def __init__(self, name, monthly_salary):
        super().__init__(name)
        self.monthly_salary = monthly_salary

    def calculate_pay(self):
        return self.monthly_salary

class Contractor(Employee):
    def __init__(self, name, hours_worked, hourly_rate):
        super().__init__(name)
        self.hours_worked = hours_worked
        self.hourly_rate = hourly_rate

    def calculate_pay(self):
        return self.hours_worked * self.hourly_rate

team = [
    FullTimeEmployee("Ananya", 75000),
    Contractor("Rahul", 120, 1200),
]

for person in team:
    print(person.name, "earns ₹", person.calculate_pay())

### What does the ABC protect us from?

Try uncommenting `employee = Employee("Meera")` below. Python raises a `TypeError`, because `Employee` has not provided the required `calculate_pay()` implementation.

A concrete child class must implement every inherited abstract method before it can be instantiated.

In [ ]:
# Uncomment to see the error:
# employee = Employee("Meera")

# This also fails because calculate_pay() is missing:
# class Intern(Employee):
#     pass
# intern = Intern("Kiran")

## 4. A customer example: a shared contract

ABCs aren't limited to employees. Suppose different customer notification channels must all support `send(message)`. We can require that method in a base class.

In [ ]:
from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def send(self, message):
        pass

class EmailNotifier(Notifier):
    def send(self, message):
        print(f"Email sent: {message}")

class SMSNotifier(Notifier):
    def send(self, message):
        print(f"SMS sent: {message}")

def notify_customer(notifier, message):
    notifier.send(message)

notify_customer(EmailNotifier(), "Your order has shipped.")
notify_customer(SMSNotifier(), "Your order is ready for pickup.")

The function `notify_customer` doesn't need to know the details of email or SMS. It only relies on the shared `send()` contract. In a real application, the method could call an email or SMS provider.

## 5. Public, protected, and private naming

Python relies mostly on naming conventions rather than Java-style enforced access modifiers.

- `name`: public; normal attribute.
- `_internal_value`: a convention meaning “internal use”; it is not truly protected.
- `__secret`: triggers name mangling, which makes accidental clashes less likely. It is not a secure way to store secrets.

Never print API keys or passwords in logs or object representations.

In [ ]:
class CustomerAccount:
    def __init__(self, name, points):
        self.name = name
        self._points = points       # Convention: internal use
        self.__account_note = "VIP" # Name-mangled attribute

    def show_points(self):
        return self._points

customer = CustomerAccount("Meera", 120)
print(customer.name)
print(customer.show_points())
# customer.__account_note  # AttributeError: name is mangled

## 6. `@classmethod` and `@staticmethod`

These are useful tools, but keep the distinction simple:

- An **instance method** receives `self` and works with one object's data.
- A **class method** receives `cls` and is often used as an alternative constructor.
- A **static method** receives neither `self` nor `cls`; it is a helper related to the class.

In [ ]:
class Customer:
    def __init__(self, name, email):
        self.name = name
        self.email = email

    @classmethod
    def from_record(cls, record):
        # Build a Customer from a dictionary
        return cls(record["name"], record["email"])

    @staticmethod
    def is_valid_email(email):
        # Simple teaching example, not full email validation
        return "@" in email and "." in email.split("@")[-1]

record = {"name": "Meera", "email": "meera@example.com"}
customer = Customer.from_record(record)
print(customer.name)
print(Customer.is_valid_email(customer.email))

## 7. Composition: HAS-A instead of IS-A

Use inheritance for an **IS-A** relationship. Use composition when one object **HAS-A** another object.

For example, a customer account has an address. An `Address` is not a kind of `Customer`; it is a separate object used by the customer.

In [ ]:
class Address:
    def __init__(self, city, country):
        self.city = city
        self.country = country

class Customer:
    def __init__(self, name, address):
        self.name = name
        self.address = address  # Customer HAS-A Address

address = Address("Hyderabad", "India")
customer = Customer("Meera", address)

print(customer.name)
print(customer.address.city)
print(customer.address.country)

## 8. Check an object's type with `isinstance()`

`isinstance(object, ClassName)` returns `True` when the object is an instance of that class or one of its subclasses.

In [ ]:
class Employee:
    pass

class FullTimeEmployee(Employee):
    pass

employee = FullTimeEmployee()

print(isinstance(employee, FullTimeEmployee))  # True
print(isinstance(employee, Employee))          # True: parent class too
print(isinstance(employee, str))               # False

## Practice tasks

1. Create a `Customer` base class with `name` and `email`; create `PremiumCustomer` that adds a discount percentage.
2. Override a method called `get_discount()` in `PremiumCustomer`.
3. Create an ABC called `PaymentMethod` with abstract method `pay(amount)`. Implement `CardPayment` and `UPIPayment`.
4. Create a function that accepts any `PaymentMethod` and calls `pay(amount)` without checking which subclass it received.
5. Add a `@classmethod` alternative constructor that creates a customer from a dictionary.
6. Model a `Customer` that HAS-A `Address` using composition.

### Quick recap
- **Inheritance:** child reuses parent behavior (IS-A).
- **Overriding:** child supplies its own version of a method.
- **`super()`:** calls parent behavior.
- **ABC / `@abstractmethod`:** define required methods that concrete subclasses must implement.
- **Composition:** one object contains or uses another (HAS-A).
- **Polymorphism:** different object types respond to the same method call in their own way.